In [190]:
import pandas as pd

path ='/Users/thomas/Data/Dynamic_weight_data/Marzo/30 - 31 marzo/1.xlsx'
df=pd.read_excel(path)


print(df.head())



len(df)

        Id      StartTime      StartTimeStr  LaneNo                  LaneName  \
0  1237982  1743290000000  29/03/2025 23:00       3  Corsia 2 - Marcia veloce   
1  1237983  1743290000000  29/03/2025 23:00       2         Corsia 1 - Marcia   
2  1237984  1743290000000  29/03/2025 23:01       2         Corsia 1 - Marcia   
3  1237985  1743290000000  29/03/2025 23:01       3  Corsia 2 - Marcia veloce   
4  1237986  1743290000000  29/03/2025 23:01       3  Corsia 2 - Marcia veloce   

   BaseClassId Scheme  ClassId  MoveStatus  FrontToFront  ...  WheelBase  \
0           21  EUR13        1           1        68.193  ...       2.68   
1           21  EUR13        1          -1        57.941  ...       2.78   
2           15  EUR13        9           1        12.547  ...      12.22   
3           21  EUR13        1           1        37.834  ...       2.76   
4           21  EUR13        1           0        40.178  ...       2.51   

   FrontOverhang  AxlesCount  MassUnit  VelocityUnit  Di

73669

In [193]:
from datetime import datetime, timedelta
import time
from tqdm import tqdm

def find_isolated_timestamps(timestamps, gap_minutes=5):
    fmt = "%d/%m/%Y %H:%M"
    
    parsed = [
        ts if isinstance(ts, datetime) else datetime.strptime(ts.strip(), fmt)
        for ts in timestamps
    ]
    parsed= sorted(parsed)

    print(parsed[2])    
    isolated = []
    gap = timedelta(minutes=gap_minutes)
    total = len(parsed)

    def find_effective_neighbor(i, direction):
        """Walk in a direction, skipping identical timestamps, return first different one."""
        j = i + direction
        while 0 <= j < total:
            if parsed[j] != parsed[i]:
                return parsed[j]
            j += direction
        return None

    for i in tqdm(range(total), desc="Scanning timestamps"):
        current = parsed[i]
        
        left_neighbor  = find_effective_neighbor(i, -1)
        right_neighbor = find_effective_neighbor(i, +1)
        
        left_ok  = left_neighbor  is None or abs(current - left_neighbor)  > gap
        right_ok = right_neighbor is None or abs(current - right_neighbor) > gap
        
        if left_ok and right_ok:
            isolated.append(current)

    print(f"\nDone — {len(isolated)} isolated timestamp(s) found out of {total}.")
    return isolated



timestamps = df['StartTimeStr']

# timestamps.sort_values(ascending=True, inplace=True)

# print(timestamps.head())

isolated = find_isolated_timestamps(timestamps, gap_minutes=2)

2025-01-04 00:01:00


Scanning timestamps: 100%|██████████| 73669/73669 [00:00<00:00, 664799.53it/s]


Done — 6 isolated timestamp(s) found out of 73669.


In [168]:
isolated = pd.to_datetime(isolated, format="%Y-%m-%d %H:%M:%S")



print(isolated)

DatetimeIndex(['2025-03-22 02:26:00', '2025-03-23 00:57:00',
               '2025-03-23 00:57:00', '2025-03-23 00:57:00',
               '2025-03-23 01:53:00', '2025-03-23 03:54:00'],
              dtype='datetime64[ns]', freq=None)


In [159]:
df_isolated_timestamps =pd.DataFrame({'timestamp': isolated})

print(df_isolated_timestamps.head())

df_isolated_timestamps['timestamp'] = df_isolated_timestamps['timestamp'].dt.strftime('%Y-%d-%m %H:%M:%S')

df_isolated_timestamps['timestamp'] = pd.to_datetime(df_isolated_timestamps['timestamp'], format="%Y-%d-%m %H:%M:%S")


print(df_isolated_timestamps.head())
print(df_isolated_timestamps.dtypes)


            timestamp
0 2025-03-18 00:15:00
1 2025-03-18 00:33:00
            timestamp
0 2025-03-18 00:15:00
1 2025-03-18 00:33:00
timestamp    datetime64[ns]
dtype: object


In [160]:



df_isolated = pd.read_csv('/Users/thomas/Desktop/github_repos/industry_time_series/src/results/less_traffic/isolated_timestamps_5min.csv', index_col=0)

print(df_isolated.dtypes)
print(df_isolated.head())

# 2. Convert to datetime
df_isolated['timestamp'] = pd.to_datetime(df_isolated['timestamp'], format="%Y-%d-%m %H:%M:%S")

print(df_isolated.head())
print(df_isolated.dtypes)


timestamp    object
dtype: object
             timestamp
0  2025-01-03 00:58:00
1  2025-01-03 01:54:00
2  2025-03-03 00:16:00
3  2025-03-03 01:21:00
4  2025-03-18 00:15:00
             timestamp
0  2025-01-03 00:58:00
1  2025-01-03 01:54:00
2  2025-03-03 00:16:00
3  2025-03-03 01:21:00
4  2025-04-03 00:25:00
5  2025-04-03 01:48:00
6  2025-06-03 00:16:00
7  2025-06-03 01:31:00
8  2025-06-03 02:19:00
9  2025-07-03 00:08:00
timestamp    datetime64[ns]
dtype: object


In [161]:

df_isolated_timestamps = pd.merge(df_isolated, df_isolated_timestamps, on='timestamp', how='outer')

print(df_isolated_timestamps.tail())

df_isolated_timestamps.to_csv('/Users/thomas/Desktop/github_repos/industry_time_series/src/results/less_traffic/isolated_timestamps_5min.csv', index=True)



             timestamp
13 2025-03-07 01:50:00
14 2025-03-08 01:48:00
15 2025-03-08 01:56:00
16 2025-03-18 00:15:00
17 2025-03-18 00:33:00
